# Ping Pong Ball Prediction Using Transfer Learning:

Inputs:
- x y z (frame 1)
- x y z (frame 2)
- x y z (frame 3)
- x y z (frame 4)
- dt12
- dt23
- dt34

Outputs:
- x y z (interception)
- vx vy vz
- t_hit
- is_reachable



## Importing the Libraries

In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

### Importing the Dataset

In [2]:
dataset = pd.read_csv('../learning_data/synthetic_data/synthetic_intercept_dataset.csv')

In [3]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 41 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   x1                     20000 non-null  float64
 1   y1                     20000 non-null  float64
 2   z1                     20000 non-null  float64
 3   x2                     20000 non-null  float64
 4   y2                     20000 non-null  float64
 5   z2                     20000 non-null  float64
 6   x3                     20000 non-null  float64
 7   y3                     20000 non-null  float64
 8   z3                     20000 non-null  float64
 9   x4                     20000 non-null  float64
 10  y4                     20000 non-null  float64
 11  z4                     20000 non-null  float64
 12  x5                     20000 non-null  float64
 13  y5                     20000 non-null  float64
 14  z5                     20000 non-null  float64
 15  x6

In [4]:
dataset.describe()

,x1,y1,z1,x2,y2,z2,x3,y3,z3,x4,...,bounces_before_hit,has_bounce_before_hit,intercept_valid,x0,y0,z0,vx0,vy0,vz0,gravity_z_mm_s2
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,...,20000.00000,20000.000000,20000.0,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.0
mean,2.846180,2586.805399,-800.782866,2.587758,2516.164890,-830.432964,2.335908,2445.474348,-857.060668,2.063134,...,1.15260,0.957000,1.0,2.846180,2586.805399,-800.782866,-16.938254,-4560.467929,-2008.859992,-9810.0
std,611.462974,375.072852,203.189690,589.374269,397.389140,201.130733,568.209203,420.543096,195.828620,548.091609,...,0.46403,0.202862,0.0,611.462974,375.072852,203.189690,2055.477303,1858.306767,1790.670399,0.0
min,-1062.362775,1940.034250,-1150.718078,-1063.103457,1765.688301,-1150.553726,-1064.714239,1589.401180,-1150.412514,-1069.299827,...,0.00000,0.000000,1.0,-1062.362775,1940.034250,-1150.718078,-5389.215897,-10000.000000,-5579.934696,-9810.0
25%,-528.535366,2260.726977,-977.935437,-509.131333,2172.421868,-1011.389874,-487.865675,2084.232867,-1040.274206,-467.281644,...,1.00000,1.000000,1.0,-528.535366,2260.726977,-977.935437,-1493.140882,-5785.656253,-3529.333450,-9810.0
50%,3.959062,2585.280814,-801.336626,1.937988,2515.546526,-832.605819,1.703397,2442.896152,-868.113493,1.632830,...,1.00000,1.000000,1.0,3.959062,2585.280814,-801.336626,-20.485697,-4299.572021,-2039.711058,-9810.0
75%,529.412526,2912.452341,-622.844420,508.848227,2861.943378,-654.383173,490.945488,2810.925219,-688.377206,474.061369,...,1.00000,1.000000,1.0,529.412526,2912.452341,-622.844420,1442.721220,-3122.030705,-599.665230,-9810.0
max,1062.362428,3239.991853,-450.793865,1062.991701,3220.357264,-424.677132,1065.423516,3204.612203,-395.711871,1067.941189,...,2.00000,1.000000,1.0,1062.362428,3239.991853,-450.793865,5259.466574,-816.493459,2414.174822,-9810.0


### Pre-Processing

### Separating the Input and Outputs

In [5]:
x = dataset.iloc[:, :15]
y = dataset.iloc[:, 15:-11]

In [6]:
# View the inputs
print(x)

               x1           y1           z1          x2           y2  \
0     -129.883316  3056.177296  -662.582380  -82.613370  3020.750138   
1     -637.695070  1949.570951  -599.892936 -598.290176  1818.696515   
2      981.532537  3121.154898  -660.945006  979.224710  3084.695960   
3     -129.813147  1968.095704  -572.335653 -156.440657  1812.347335   
4     -403.062195  2950.626290  -470.461502 -415.733914  2903.238033   
...           ...          ...          ...         ...          ...   
19995 -569.758900  1956.017896  -541.428463 -548.070337  1834.083464   
19996 -576.209725  1945.075686  -657.666635 -542.943741  1825.041555   
19997   -0.633386  2788.299177  -802.618530   18.380385  2696.490869   
19998  928.145514  3204.306094 -1082.108162  877.262426  3156.054259   
19999  584.466916  2829.469475  -779.738803  538.987805  2785.366693   

                z2          x3           y3           z3          x4  \
0      -652.486762  -39.135189  2988.164773  -645.857170    6.7

In [7]:
# View the outputs
print(y)

               x6           y6           z6      dt12      dt23      dt34  \
0       85.426327  2894.810306  -641.162618  0.017634  0.016219  0.017114   
1     -443.534728  1304.711322  -753.980682  0.016342  0.015355  0.015826   
2      971.355387  2960.376792  -702.029303  0.017513  0.015279  0.014012   
3     -248.855290  1271.800047  -908.543422  0.016843  0.013539  0.017580   
4     -467.456775  2709.810758  -826.923616  0.015532  0.016471  0.015906   
...           ...          ...          ...       ...       ...       ...   
19995 -472.144009  1407.220989  -781.746239  0.017707  0.013533  0.016670   
19996 -416.033935  1367.111065  -861.822858  0.016500  0.017215  0.014387   
19997   86.665441  2366.775362 -1094.927325  0.017530  0.017862  0.016261   
19998  658.834572  2948.921688  -857.125388  0.014875  0.014765  0.016322   
19999  363.911134  2615.588345  -747.564670  0.015927  0.017044  0.016002   

           dt45      dt56        x_hit        y_hit        z_hit       vx_h

In [8]:
x = x.values
y = y.values

### Splitting the Dataset into the Training set and Test set

In [9]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1) Build reachability vector from your full dataset
# (change column name if needed)
r = dataset["is_reachable"].to_numpy().astype(int)

# 2) Split x, y, and r together (important)
x_train, x_test, y_train, y_test, r_train, r_test = train_test_split(
    x, y, r,
    test_size=0.05,
    random_state=1,
    stratify=r   # recommended
)

# 3) Keep full test copy
x_test_all = x_test.copy()
y_test_all = y_test.copy()

# 4) Reachable-only adjusted test set
mask_reach = (r_test == 1)
x_test_reach = x_test_all[mask_reach].copy()
y_test_reach = y_test_all[mask_reach].copy()


In [10]:
print(x_train.shape)
print(x_test_all.shape)
print(x_test_reach.shape)
print(y_train.shape)
print(y_test_all.shape)
print(y_test_reach.shape)

(19000, 15)
(1000, 15)
(236, 15)
(19000, 15)
(1000, 15)
(236, 15)


### Feature Scaling

In [11]:
from sklearn.preprocessing import StandardScaler

scx = StandardScaler()
scy = StandardScaler()

x_train = scx.fit_transform(x_train)
x_test_all = scx.transform(x_test_all)
x_test_reach = scx.transform(x_test_reach)

y_train = scy.fit_transform((y_train))
y_test_all = scy.transform((y_test_all))
y_test_reach = scy.transform((y_test_reach))

## The ANN Model Initialization

In [12]:
ball_predictor = tf.keras.models.Sequential()

### Constructing the Model

In [13]:
ball_predictor.add(tf.keras.layers.Dense(units=15, activation='relu')) # 15 input features
ball_predictor.add(tf.keras.layers.Dense(units=30, activation='relu')) # hiddel layer 1
ball_predictor.add(tf.keras.layers.Dense(units=15, activation='relu')) # hidden layer 3
ball_predictor.add(tf.keras.layers.Dense(units=7, activation='linear')) # 7

In [14]:
ball_predictor.compile(loss='mean_squared_error', optimizer='adam', metrics=['mean_squared_error'])

### Training

In [15]:
p1 = ball_predictor.fit(x_train, y_train, validation_split=0.2, batch_size = 32, epochs = 300,verbose=1)

Epoch 1/300


ValueError: Dimensions must be equal, but are 15 and 7 for '{{node compile_loss/mean_squared_error/sub}} = Sub[T=DT_FLOAT](data_1, sequential_1/dense_3_1/BiasAdd)' with input shapes: [32,15], [32,7].

### Learning Curves

In [ ]:
fig, axs = plt.subplots(figsize=(16,5))
fig.suptitle('Loss Function')

axs.plot(p1.history['loss'])
axs.plot(p1.history['val_loss'])
axs.set_title("Ball Prediction Learning Curves")
axs.set(xlabel='Epoch', ylabel='Loss')

fig.legend(labels=['Train', 'Validation'], loc="upper left")
plt.show()

In [ ]:
# def eval_regression(name, y_true, y_pred):
#     mae = mean_absolute_error(y_true, y_pred, multioutput='raw_values')
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred, multioutput='raw_values'))
#     print(f"\n{name}")
#     print("MAE per output:", mae)
#     print("RMSE per output:", rmse)

# # If y is scaled, inverse-transform both y_true and y_pred before metrics.
# y_pred_all = ball_predictor.predict(x_test_all)
# y_pred_reach = ball_predictor.predict(x_test_reach)

# # Example for scaled targets:
# # y_pred_all = scy.inverse_transform(y_pred_all); y_test_all_eval = scy.inverse_transform(y_test_all)
# # y_pred_reach = scy.inverse_transform(y_pred_reach); y_test_reach_eval = scy.inverse_transform(y_test_reach)

# eval_regression("All test shots", y_test_all, y_pred_all)
# eval_regression("Reachable-only test shots", y_test_reach, y_pred_reach)

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

target_names = ["x_hit", "y_hit", "z_hit", "vx_hit", "vy_hit", "vz_hit", "t_hit"]
target_units = ["mm", "mm", "mm", "mm/s", "mm/s", "mm/s", "ms"]

def eval_with_units(name, model, x_eval, y_eval, y_scaler, y_is_scaled=True, t_hit_in_seconds=True):
    y_pred = model.predict(x_eval, verbose=0)

    # Convert to physical units exactly once
    if y_is_scaled:
        assert y_eval.shape[1] == y_scaler.n_features_in_ == len(target_names)
        y_true_u = y_scaler.inverse_transform(y_eval)
        y_pred_u = y_scaler.inverse_transform(y_pred)
    else:
        y_true_u = y_eval
        y_pred_u = y_pred

    err = y_pred_u - y_true_u
    abs_err = np.abs(err)

    mae = mean_absolute_error(y_true_u, y_pred_u, multioutput="raw_values")
    rmse = np.sqrt(mean_squared_error(y_true_u, y_pred_u, multioutput="raw_values"))
    std_abs = abs_err.std(axis=0)

    # Convert t_hit to ms for reporting (only if stored in seconds)
    if t_hit_in_seconds:
        i = target_names.index("t_hit")
        mae[i] *= 1000.0
        rmse[i] *= 1000.0
        std_abs[i] *= 1000.0

    report = pd.DataFrame({
        "target": target_names,
        "unit": target_units,
        "MAE": mae,
        "RMSE": rmse,
        "STD_abs_error": std_abs,
    })

    print(f"\n{name}")
    print(report.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

# Run both:
eval_with_units("All test shots", ball_predictor, x_test_all, y_test_all, scy, y_is_scaled=True, t_hit_in_seconds=True)
eval_with_units("Reachable-only test shots", ball_predictor, x_test_reach, y_test_reach, scy, y_is_scaled=True, t_hit_in_seconds=True)



In [ ]:
# Verification 
train_std = scy.scale_ 
rmse_sigma = np.sqrt(0.1084)   # val_mse from curve, e.g. 0.1151
rmse_units = rmse_sigma * train_std

print(rmse_units)


## Loading the Real Dataset (Transfer Learning)

In [ ]:
real_dataset = pd.read_csv('../learning_data/real_data/real_transfer_dataset_clean.csv')
real_dataset.info()

In [ ]:
real_dataset.describe()

### Pre-Processing the Real Dataset

In [ ]:
x_real = real_dataset.iloc[:, 2:17]
y_real = real_dataset.iloc[:, 17:-3]

In [ ]:
# View the inputs
print(x_real)

In [ ]:
# View the inputs
print(y_real)

In [ ]:
x_real = x_real.values
y_real = y_real.values

### Splitting the Dataset into the Training set and Test set

In [ ]:
#Split input and output x, y
x_real_train, x_real_test, y_real_train, y_real_test= train_test_split(
    x_real, y_real,
    test_size=0.1,
    random_state=1
)


In [ ]:
print(x_real_train.shape)
print(x_real_test.shape)
print(y_real_train.shape)
print(y_real_test.shape)

### Scaling the real dataset

In [ ]:
x_real_train = scx.transform(x_real_train)
x_real_test = scx.transform(x_real_test)
y_real_train = scy.transform(y_real_train)
y_real_test = scy.transform(y_real_test)

## Training the New Layers (Classification Layers)

In [ ]:
from sklearn.model_selection import KFold
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import numpy as np
import pandas as pd

# run this in a new cell BEFORE Stage 1
fold_states = []
stage1_reports = []
stage2_reports = []


target_names = ["x_hit","y_hit","z_hit","vx_hit","vy_hit","vz_hit","t_hit"]
target_units = ["mm","mm","mm","mm/s","mm/s","mm/s","ms"]

def metrics_df(model, x_eval_s, y_eval_s, scy, t_hit_in_seconds=True):
    y_pred_s = model.predict(x_eval_s, verbose=0)
    y_true_u = scy.inverse_transform(y_eval_s)
    y_pred_u = scy.inverse_transform(y_pred_s)

    err = y_pred_u - y_true_u
    abs_err = np.abs(err)

    mae = abs_err.mean(axis=0)
    rmse = np.sqrt((err**2).mean(axis=0))
    std_abs = abs_err.std(axis=0)

    if t_hit_in_seconds:
        i = target_names.index("t_hit")
        mae[i] *= 1000.0
        rmse[i] *= 1000.0
        std_abs[i] *= 1000.0

    return pd.DataFrame({
        "target": target_names, "unit": target_units,
        "MAE": mae, "RMSE": rmse, "STD_abs_error": std_abs
    })

kf = KFold(n_splits=5, shuffle=True, random_state=1)
fold_states = []
stage1_reports = []

for fold, (tr_idx, te_idx) in enumerate(kf.split(x_real), 1):
    x_tr, x_te = x_real[tr_idx], x_real[te_idx]
    y_tr, y_te = y_real[tr_idx], y_real[te_idx]

    x_tr_s = scx.transform(x_tr); x_te_s = scx.transform(x_te)
    y_tr_s = scy.transform(y_tr); y_te_s = scy.transform(y_te)

    model = tf.keras.models.clone_model(ball_predictor)
    model.set_weights(ball_predictor.get_weights())

    for layer in model.layers[:2]:
        layer.trainable = False

    model.compile(loss="mean_squared_error", optimizer=Adam(5e-4), metrics=["mean_squared_error"])
    history_stage1 = model.fit(x_tr_s, y_tr_s, validation_split=0.2, epochs=100, verbose=1)

    fold_states.append({
        "fold": fold,
        "model": model,
        "x_train": x_tr_s, "y_train": y_tr_s,
        "x_test": x_te_s, "y_test": y_te_s,
        "history_stage1": history_stage1.history
    })



    rep = metrics_df(model, x_te_s, y_te_s, scy, t_hit_in_seconds=True)
    rep["fold"] = fold
    stage1_reports.append(rep)


stage1_all = pd.concat(stage1_reports, ignore_index=True)
stage1_summary = stage1_all.groupby(["target","unit"], as_index=False).agg(
    MAE_mean=("MAE","mean"), MAE_std=("MAE","std"),
    RMSE_mean=("RMSE","mean"), RMSE_std=("RMSE","std")
)

print("Stage 1 K-fold summary:")
print(stage1_summary.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))



## Fine-Tuning the Early Layers (Feature Extraction Layers)

In [ ]:
import numpy as np
train_curves = np.array([s["history_stage1"]["loss"] for s in fold_states])
val_curves   = np.array([s["history_stage1"]["val_loss"] for s in fold_states])

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(train_curves.mean(axis=0), label="Train mean")
ax.plot(val_curves.mean(axis=0), label="Validation mean")
ax.set_title("Stage 1 - Mean K-fold Curves")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
plt.show()


In [ ]:
#evaluation
print("=== Stage 1: per-fold ===")
print(stage1_all.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

print("\n=== Stage 1: K-fold average ± std ===")
print(stage1_summary.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))


In [ ]:
print("fold_states len:", len(fold_states))
print("fold ids:", [s["fold"] for s in fold_states])


In [ ]:
stage2_reports = []

for state in fold_states:
    model = state["model"]  # continue from stage-1 weights

    for layer in model.layers:
        layer.trainable = True

    model.compile(loss="mean_squared_error", optimizer=Adam(1e-5), metrics=["mean_squared_error"])
    history_stage2 = model.fit(state["x_train"], state["y_train"], validation_split=0.2, epochs=100, verbose=1)

    state["history_stage2"] = history_stage2.history


    rep = metrics_df(model, state["x_test"], state["y_test"], scy, t_hit_in_seconds=True)
    rep["fold"] = state["fold"]
    stage2_reports.append(rep)

stage2_all = pd.concat(stage2_reports, ignore_index=True)
stage2_summary = stage2_all.groupby(["target","unit"], as_index=False).agg(
    MAE_mean=("MAE","mean"), MAE_std=("MAE","std"),
    RMSE_mean=("RMSE","mean"), RMSE_std=("RMSE","std")
)

print("Stage 2 K-fold summary:")
print(stage2_summary.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))


In [ ]:
import numpy as np
train_curves = np.array([s["history_stage2"]["loss"] for s in fold_states])
val_curves   = np.array([s["history_stage2"]["val_loss"] for s in fold_states])

fig, ax = plt.subplots(figsize=(12,4))
ax.plot(train_curves.mean(axis=0), label="Train mean")
ax.plot(val_curves.mean(axis=0), label="Validation mean")
ax.set_title("Stage 2 - Mean K-fold Curves")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.legend()
plt.show()


In [ ]:
print("=== Stage 2: per-fold ===")
print(stage2_all.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))

print("\n=== Stage 2: K-fold average ± std ===")
print(stage2_summary.to_string(index=False, float_format=lambda v: f"{v:,.3f}"))
